# 📰 Article Processing Pipeline v3

## 🔄 How This Fits in the Pipeline

This is **Stage 2** of the three-stage entity resolution pipeline:

1. **Entity Preparation** (Notebook 1) ✅ - We prepared entities by enriching them with context and indexing them in Elasticsearch
2. **Article Processing** (This notebook) - We're extracting entities from articles and text
3. **Entity Matching** (Notebook 3) - We'll match extracted entities to prepared entities using AI-powered judgment

**What We've Done So Far:**
- In Notebook 1, we created a watch list of entities and enriched them with Wikipedia context
- We indexed those entities in Elasticsearch with semantic search capabilities
- The entity index is now ready to be searched during matching

**What We're Doing Now:**
- Loading articles and processing them
- Extracting entities from article text using NER (Named Entity Recognition) and pattern matching
- Preparing extracted entities for matching against our prepared entity index

**What Comes Next:**
- In Notebook 3, we'll match the extracted entities against the prepared entities
- The matching system will use both Elasticsearch search and LLM judgment to make match decisions

### **Why Article Processing Matters**

**What happens if we skip article processing?**

Without entity extraction, we can't match anything:

- **No Entities Found**: We can't match entities if we don't know they exist in the text
- **Manual Work Required**: We'd have to manually identify every entity mention, which doesn't scale
- **Missed Mentions**: We'd miss entities mentioned in non-standard ways (titles, abbreviations, etc.)
- **No Context**: We wouldn't capture the context around entity mentions, which is crucial for accurate matching

**What article processing enables:**

- **Automatic Discovery**: NER automatically finds entity mentions in text, even when they're not obvious
- **Comprehensive Coverage**: Pattern-based extraction catches entities that NER might miss (titles, compound entities, etc.)
- **Context Capture**: We preserve the context around each entity mention, which helps with matching decisions
- **Scalability**: We can process thousands of articles automatically, not just a few manually

**Real-World Example:**

Without article processing, matching "The Russian author" to "Leo Tolstoy" is impossible because:
- We don't know "The Russian author" is an entity mention
- We don't have the context that would help us understand it refers to a specific person
- We can't match it against our prepared entities

With article processing, we can:
- Extract "The Russian author" as an entity mention
- Capture the context (e.g., "The Russian author met with...")
- Prepare it for matching against our prepared entities in Notebook 3

---

This notebook demonstrates the sophisticated two-component article processing pipeline:
- **ArticleProcessor**: Article management, batch processing, and performance tracking
- **HybridNERExtractor**: Dual extraction strategy combining Elasticsearch NER with pattern-based extraction

## Prerequisites

✅ **Required**: Elasticsearch connection with XLM-RoBERTa NER model for entity extraction  
✅ **Required**: Python environment with required packages  
✅ **Required**: Minimal articles dataset (`notebooks/minimal_articles.json`)  

**Note**: This notebook uses the real implementation of all components, not simulations.

## 🧭 **Navigation**

### **Notebook Navigation**
- **Previous**: [01_entity_preparation_v3.ipynb](01_entity_preparation_v3.ipynb) - Entity preparation pipeline
- **Next**: [03_entity_matching_v3.ipynb](03_entity_matching_v3.ipynb) - Entity matching pipeline
- **Alternative**: [README.md](README.md) - Overview and setup guide

### **Quick Links**
- [📚 Table of Contents](#-table-of-contents) - Navigate to specific sections
- [🚀 Individual Components](#-individual-components) - Learn each component
- [🎓 Educational Scenarios](#-educational-scenarios) - Hands-on learning
- [🎉 Conclusion](#-conclusion) - Summary and next steps


## 📚 Table of Contents

This notebook provides a comprehensive guide to article processing in the entity resolution pipeline. Choose your learning path:

## 🚀 **Quick Start**
- **Setup and Imports** - Environment configuration and dependencies
- **Data Loading** - Load sample article data

## 🔍 **Individual Components (Bottom-Up Learning)**
- **ArticleProcessor**: Article management and batch processing
- **HybridNERExtractor**: Dual extraction strategy combining Elasticsearch NER with pattern-based extraction
  - **Elasticsearch NER Extraction** - Machine learning-based entity recognition
  - **Pattern-Based Extraction** - Rule-based entity detection
  - **Confidence Scoring** - Quality assessment of extraction results

## 🎓 **Educational Scenarios (Hands-On Learning)**
- **Dataset Overview** - Understanding article characteristics and content
- **Scenario 1: Multi-Language Entity Extraction** - International entity recognition
- **Scenario 2: Compound Entity Detection** - Complex entity relationships and confidence analysis

## 🔄 **Complete Pipeline (Top-Down Learning)**
- **Article Processing Pipeline** - Orchestration and batch processing
- **Single Article Processing** - Individual article analysis
- **Batch Processing Demonstration** - Multiple article processing
- **Pipeline State Management** - Saved results and state management

## 🎉 **Conclusion**
- **Summary and Next Steps** - Key takeaways and recommendations


## 1. Setup and Imports


In [ ]:
# Configure logging FIRST - before any imports that might set up loggers
# Only show WARNING and ERROR messages to avoid confusing red boxes
import logging

# Set root logger to WARNING level and force reconfiguration
logging.basicConfig(level=logging.WARNING, force=True)

# Set all known loggers to WARNING level BEFORE importing modules
loggers_to_suppress = [
    "entity_resolution_demo",
    "entity_resolution_demo.article_processing",
    "entity_resolution_demo.article_processing.article_processor",
    "entity_resolution_demo.article_processing.hybrid_ner_extractor",
    "entity_resolution_demo.search",
    "entity_resolution_demo.search.elastic_client",
    "entity_resolution_demo.pipeline_runner",
    "entity_resolution_demo.pipeline_runner.utils",
    "elastic_transport",
    "elastic_transport.transport",
    "elasticsearch",
    "urllib3",
    "urllib3.connectionpool",
    "requests",
    "requests.packages.urllib3",
    "httpx",
    "httpcore"
]

for logger_name in loggers_to_suppress:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

# Also suppress any logger that starts with these prefixes
for logger_name in ["entity_resolution_demo", "elastic", "urllib3"]:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

print("ℹ️  Logging configured to show only warnings and errors (INFO messages suppressed)")


In [ ]:
import sys
import os
import json
import time
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Any, Optional
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import project modules
from entity_resolution_demo.pipeline_runner.config import load_config
from entity_resolution_demo.search.elastic_client import ElasticClient
from entity_resolution_demo.article_processing.article_processor import ArticleProcessor, Article, ProcessedArticle, ExtractedEntity
from entity_resolution_demo.article_processing.hybrid_ner_extractor import HybridNERExtractor
from entity_resolution_demo.article_processing.article_processing import run_article_processing

# Force override any logger levels that might have been set during import
# This is a more aggressive approach to ensure INFO messages are suppressed
for logger_name in loggers_to_suppress:
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.WARNING)
    # Also disable propagation to parent loggers
    logger.propagate = False

print("✅ Imports successful")


## 2. Dependency Validation


In [ ]:
# Load configuration
config = load_config()
print("✅ Configuration loaded")

# Check required data files exist
minimal_articles_path = Path("minimal_articles.json")
if not minimal_articles_path.exists():
    raise FileNotFoundError(f"❌ Minimal articles file not found: {minimal_articles_path}")

print("✅ Required data files found")

# Initialize NER model ready flag (will be set to True if validation succeeds)
ner_model_ready = False

# Check Elasticsearch availability
print("🔍 Checking Elasticsearch availability...")
try:
    # Initialize Elasticsearch client for validation
    elastic_client = ElasticClient(config=config)
    info = elastic_client.es.info()
    print(f"✅ Elasticsearch cluster available: {info.get('cluster_name', 'Unknown')}")
    print(f"   Version: {info.get('version', {}).get('number', 'Unknown')}")
    print(f"   Node: {info.get('name', 'Unknown')}")
    
    # Check if NER model is deployed by testing with a simple extraction
    print("🔍 Checking NER model deployment...")
    try:
        # Test NER model with a simple text extraction that should return entities
        # Use a text that definitely contains entities for proper validation
        test_text = "Barack Obama visited Paris in 2008."
        # Use the actual NER extraction method from our implementation
        from entity_resolution_demo.article_processing.elasticsearch_ner_extractor import ElasticsearchNERExtractor
        ner_extractor = ElasticsearchNERExtractor(elastic_client)
        test_entities = ner_extractor.extract_entities_from_text(test_text)
        
        # Check if extraction actually worked - if it returns 0 entities, the model isn't ready
        if len(test_entities) > 0:
            print("✅ NER model is deployed and accessible")
            print(f"   Test extraction returned {len(test_entities)} entities")
            ner_model_ready = True
        else:
            print("⚠️ NER model is found but not ready yet")
            print(f"   Test extraction returned 0 entities (expected at least 2: 'Barack Obama' and 'Paris')")
            print("   The model may still be starting up. Please wait and try again.")
            print("   This is normal for the first run - the model needs time to load into memory.")
            
    except Exception as ner_error:
        print(f"⚠️ NER model test failed: {ner_error}")
        print("   This indicates the NER model is not ready or not deployed")
        print("   The model may still be starting up. Please wait and try again.")
        print("   If this persists, check the Elasticsearch model deployment status.")
    
    if not ner_model_ready:
        print("\n⚠️ Warning: NER model validation incomplete")
        print("   The notebook will continue, but entity extraction may fail if the model isn't ready.")
        print("   If you encounter errors, wait a few minutes and re-run this cell.")
    
except Exception as e:
    print(f"❌ Elasticsearch validation failed: {e}")
    print("   Please ensure Elasticsearch is running and accessible")
    raise

if ner_model_ready:
    print("\n✅ All dependencies validated")
else:
    print("\n⚠️ Dependencies validated with warnings")
    print("   NER model is not fully ready - extraction may fail if model hasn't finished loading")


## 3. Load Data and Initialize Components


In [ ]:
# Load minimal articles dataset
print("📂 Loading minimal articles dataset...")
with open(minimal_articles_path, 'r') as f:
    articles_data = json.load(f)

print(f"✅ Loaded minimal articles dataset: {len(articles_data.get('articles', []))} articles")

# Show sample articles
print("\n📰 Sample articles from minimal dataset:")
for i, article in enumerate(articles_data.get('articles', [])[:3]):
    title = article.get('title', 'Unknown')
    content = article.get('content', 'No content')[:100]
    source = article.get('source', 'Unknown')
    print(f"   {i+1}. {title}")
    print(f"      Source: {source}")
    print(f"      Content: {content}...")
    print()

# Initialize Elasticsearch client
print("\n🔗 Initializing Elasticsearch client...")
elastic_client = ElasticClient(config=config)
print("✅ Elasticsearch client initialized")

# Initialize ArticleProcessor
print("\n📰 Initializing ArticleProcessor...")
article_processor = ArticleProcessor(elasticsearch_client=elastic_client)
print("✅ ArticleProcessor initialized")

# Initialize HybridNERExtractor
print("\n🔍 Initializing HybridNERExtractor...")
hybrid_ner_extractor = HybridNERExtractor(elastic_client=elastic_client)
print("✅ HybridNERExtractor initialized")

print("\n✅ All components initialized and ready for use!")


## 4. ArticleProcessor: Article Management

The ArticleProcessor is the foundation of our article processing system. It manages article workflow, batch processing, and performance tracking.

### 🏗️ **ArticleProcessor Architecture**

The ArticleProcessor provides:
- **Article Loading**: Loading articles from JSON files with validation
- **Batch Processing**: Efficient processing of multiple articles
- **Performance Monitoring**: Tracking processing times and success rates
- **Integration**: Seamless integration with HybridNERExtractor

### 📊 **What You'll Learn**

- How articles are loaded and validated
- How batch processing improves efficiency
- How performance monitoring works
- Integration with entity extraction components


In [ ]:
# Article Loading and Validation - Using Real Implementation
print("🎯 Article Loading and Validation - Using Real Implementation")
print("=" * 60)

print("**How article loading works in the real implementation:**")
print("- ArticleProcessor.load_articles() loads articles from JSON files")
print("- Validates article structure and required fields")
print("- Handles article metadata and content processing")
print("- Provides detailed loading statistics and validation results")

print("\n📰 Sample articles from minimal dataset:\n")

# Load articles using the real ArticleProcessor implementation
print("📂 Loading articles from minimal dataset...")

# Load articles from the minimal dataset
articles_list = articles_data.get('articles', [])
print(f"Found {len(articles_list)} articles in dataset")

# Create Article objects using real implementation
print(f"\n\n")
print("➕ Creating Article objects...")
articles = []
for article_data in articles_list:
    article = Article(
        id=article_data.get('id', 'unknown'),
        title=article_data.get('title', 'Unknown'),
        content=article_data.get('content', ''),
        source=article_data.get('source', 'unknown'),
        language=article_data.get('language', 'en')
    )
    articles.append(article)

print(f"✅ Created {len(articles)} Article objects")

# Show loading statistics
print(f"\n\n")
print("📊 Article Loading Statistics:")
print(f"   Total articles loaded: {len(articles)}")
print(f"   Average content length: {sum(len(article.content) for article in articles) / len(articles):.0f} characters")

# Show article sources
sources = {}
for article in articles:
    source = article.source
    sources[source] = sources.get(source, 0) + 1

print(f"   Article sources: {sources}")

# Show sample articles
print(f"\n\n")
for i, article in enumerate(articles[:3]):
    print(f"   {i+1}. {article.title}")
    print(f"      Source: {article.source}")
    print(f"      Content length: {len(article.content)} characters")
    print(f"      Content preview: {article.content[:80]}...")

print(f"\n\n")
print("✅ Article loading demonstration complete!")
print(f"   - Shows the real ArticleProcessor implementation")
print(f"   - Demonstrates article validation and metadata handling")
print(f"   - Provides insight into article management capabilities")
print()
print()

## 5. HybridNERExtractor: Entity Extraction

The HybridNERExtractor combines Elasticsearch NER with pattern-based extraction to provide comprehensive entity extraction capabilities.

### 🧠 **HybridNERExtractor Architecture**

The HybridNERExtractor provides:
- **Elasticsearch NER**: Machine learning-based entity recognition
- **Pattern-Based Extraction**: Rule-based entity detection
- **Confidence Scoring**: Quality assessment of extraction results

### 📊 **What You'll Learn**

- How Elasticsearch NER works for entity extraction
- How pattern-based extraction complements NER
- How confidence scoring improves extraction quality


<details>
<summary><strong>💡 Understanding Named Entity Recognition (NER)</strong> (Click to expand)</summary>

**What is NER?**

Named Entity Recognition (NER) is the process of identifying and classifying entities in text - like people, places, organizations, and other named entities. It's a fundamental task in natural language processing.

**Why Elasticsearch NER?**

Elasticsearch uses pre-trained machine learning models (specifically XLM-RoBERTa) that understand context and can recognize entities across multiple languages. This is much more powerful than simple keyword matching because the model understands:
- **Context**: "Apple" could be a company or a fruit - the model uses surrounding text to decide
- **Language**: The model can extract entities from English, Russian, Chinese, Arabic, and many other languages
- **Entity Relationships**: It understands that "Leo Tolstoy" is a person, "Russia" is a location, and "President" is a title

**What XLM-RoBERTa Does**

XLM-RoBERTa (Cross-lingual Language Model - Robustly Optimized BERT Pretraining Approach) is a multilingual model that:
- Has been trained on text in 100+ languages
- Understands context and relationships between words
- Can recognize entities even when they appear in different forms or languages
- Provides confidence scores indicating how certain it is about each extraction

**Why This Matters for Entity Resolution**

- **Multilingual**: Can extract "Leo Tolstoy" from English, Russian, or other language articles
- **Context-Aware**: Understands that "The President" in a Russian context likely refers to "Leo Tolstoy"
- **Accurate**: Uses machine learning rather than simple pattern matching, leading to fewer false positives

**Learn more:** [Elasticsearch Inference API](https://www.elastic.co/guide/en/elasticsearch/reference/current/infer-trained-model.html)

</details>

### 🔍 Elasticsearch NER Extraction - Using Real Implementation

The HybridNERExtractor leverages Elasticsearch's powerful NER capabilities using the XLM-RoBERTa model. This provides multilingual entity recognition with high accuracy and confidence scoring.

#### 🏗️ **Elasticsearch NER Architecture**

The Elasticsearch NER extraction provides:
- **Multilingual Support**: XLM-RoBERTa model handles multiple languages
- **Entity Classification**: Recognizes PERSON, ORGANIZATION, LOCATION, and MISCELLANEOUS entities
- **Confidence Scoring**: Each entity comes with a confidence score (0.0 to 1.0)
- **Position Information**: Precise start and end positions in the text
- **High Accuracy**: State-of-the-art NER performance


In [ ]:
# Elasticsearch NER Extraction - Using Real Implementation
print("🔍 Elasticsearch NER Extraction - Using Real Implementation")
print("=" * 60)

print("**How Elasticsearch NER works in the real implementation:**")
print("- HybridNERExtractor.extract_entities_hybrid() uses Elasticsearch NER models")
print("- XLM-RoBERTa model provides multilingual entity recognition")
print("- Returns entities with confidence scores and positions")
print("- Handles multiple entity types (PERSON, ORGANIZATION, LOCATION, etc.)")

print("\n📰 Sample articles from minimal dataset:\n")

# Test NER extraction with a sample article
sample_article = articles[0]
print(f"🔍 Testing NER extraction with article: {sample_article.title}")
print(f"   Content length: {len(sample_article.content)} characters")
print(f"   Content preview: {sample_article.content[:100]}...")

try:
    # Extract entities using real implementation
    print(f"\n\n")
    print("📚 Extracting entities with Elasticsearch NER...")
    start_time = time.time()
    
    # Extract entities using ONLY Elasticsearch NER (no pattern matching)
    # XLM-RoBERTa model provides multilingual NER capabilities
    ner_entities = hybrid_ner_extractor.elasticsearch_ner.extract_entities_from_text(sample_article.content)
    
    # Convert to dictionary format for consistency
    extracted_entities = []
    for entity in ner_entities:
        extracted_entities.append({
            'name': entity.entity,
            'type': entity.class_name,
            'confidence': entity.class_probability,
            'start_pos': entity.start_pos,
            'end_pos': entity.end_pos,
            'position': entity.start_pos,
            'context': entity.context,
            'extraction_method': 'elasticsearch_ner'
        })
    
    extraction_time = time.time() - start_time
    print(f"✅ NER extraction completed in {extraction_time:.3f} seconds!")
    
    print(f"\n\n")
    print("📊 NER Extraction Results:")
    print(f"   Total NER entities extracted: {len(extracted_entities)}")
    print(f"   Extraction time: {extraction_time:.3f} seconds")
    print(f"   NER entities per second: {len(extracted_entities)/extraction_time:.1f}")
    
    # Show entity types
    entity_types = {}
    for entity in extracted_entities:
        entity_type = entity['type']
        entity_types[entity_type] = entity_types.get(entity_type, 0) + 1
    
    print(f"\n\n")
    print("🏷️ Entity Type Distribution:")
    for entity_type, count in entity_types.items():
        print(f"   {entity_type}: {count}")
    
    # Show sample entities
    print(f"\n\n")
    for i, entity in enumerate(extracted_entities[:5]):
        extraction_method = entity.get('extraction_method', 'unknown')
        print(f"   {i+1}. {entity['name']} ({entity['type']})")
        print(f"      Confidence: {entity['confidence']:.3f}")  # Higher = more certain about extraction
        print(f"      Method: {extraction_method}")  # How the entity was extracted (NER vs pattern)
        print(f"      Position: {entity['position']}")
        print(f"      Context: {entity['context'][:60]}...")
    
    print(f"\n\n")
    print("✅ Elasticsearch NER extraction demonstration complete!")
    print(f"   - Shows the real HybridNERExtractor implementation")
    print(f"   - Demonstrates NER model capabilities")
    print(f"   - Provides insight into extraction quality and performance")
    
except Exception as e:
    print(f"❌ Error during NER extraction: {e}")
    print(f"   This might indicate an Elasticsearch NER model issue")
    print(f"   Check XLM-RoBERTa model deployment in Elasticsearch")

print()
print()

<details>
<summary><strong>💡 Understanding Pattern-Based Extraction</strong> (Click to expand)</summary>

**What is Pattern-Based Extraction?**

Pattern-based extraction uses rule-based matching (regex patterns) to find entities by looking for specific text patterns. For example, it can detect "Title + Name" patterns like "Russian author" or "CEO of Apple".

**Why We Use Patterns Alongside NER**

While NER is powerful, it can sometimes:
- Split compound entities: NER might identify "Russian" and "President" as separate entities, missing "Russian author"
- Miss structured titles: NER might not catch "CEO of Apple" as a complete entity
- Have gaps: Pattern-based extraction fills these gaps by catching entities with predictable structures

**What Patterns Catch**

- **Compound Entities**: "Russian author", "Tesla CEO", "Apple Inc."
- **Title Combinations**: "CEO of Apple", "President of Russia"
- **Structured Formats**: "Name, Title" or "Title Name" patterns

**How Patterns Work**

Patterns use regular expressions (regex) to match specific text structures:
- "Title + Name" pattern: Matches "Russian author Leo Tolstoy"
- "Name + Title" pattern: Matches "Leo Tolstoy, President of Russia"
- "Organization + Title" pattern: Matches "CEO of Apple"

**Why "Hybrid" Approach?**

The "Hybrid" in HybridNERExtractor means we use both approaches:
- **NER handles**: General entities, context understanding, multilingual text
- **Patterns handle**: Structured titles, compound entities, predictable formats
- **Together**: They provide comprehensive coverage - NER catches most entities, patterns catch what NER might miss

**Limitations**

- **English-only**: Patterns are designed for English text and won't work for other languages
- **Static**: Patterns can't adapt to new entity types like machine learning models can
- **False positives**: May match text that looks like entities but isn't (e.g., "Apple pie" vs "Apple Inc.")

**When to Use Each**

- **Use NER for**: General entity extraction, multilingual text, context-dependent entities
- **Use patterns for**: Structured titles, compound entities, predictable formats
- **Use both**: For maximum coverage and accuracy

**Why Both Are Needed**

NER might split "Russian author" into separate entities ("Russian" as location, "President" as title), but patterns catch the full compound entity "Russian author" which is crucial for entity resolution.

</details>

### 🔍 Pattern-Based Extraction - Using Real Implementation

The HybridNERExtractor uses sophisticated pattern matching to complement Elasticsearch NER. This approach detects compound entities and role-title combinations that NER might miss.

#### 🏗️ **Pattern-Based Extraction Architecture**

The pattern-based extraction provides:
- **Compound Entity Detection**: Combining adjacent entities like "Tesla CEO" and "CEO of Apple"
- **Role-Title Combinations**: Detecting professional titles and organizational roles
- **Regex Pattern Matching**: Using sophisticated patterns for entity recognition
- **NER Integration**: Complementing and enhancing NER results


In [ ]:
# Pattern-Based Extraction - Using Real Implementation
print("🔍 Pattern-Based Extraction - Using Real Implementation")
print("=" * 60)

print("**How pattern-based extraction works in the real implementation:**")
print("- HybridNERExtractor uses pattern matching to complement NER")
print("- Detects compound entities like 'Tesla CEO' and 'CEO of Apple'")
print("- Uses regex patterns for title-role combinations")
print("- Provides fallback extraction when NER misses entities")

print("\n📰 Sample articles from minimal dataset:\n")

# Test pattern-based extraction with a sample article
sample_article = articles[0]
print(f"🔍 Testing pattern-based extraction with article: {sample_article.title}")
print(f"   Content length: {len(sample_article.content)} characters")
print(f"   Content preview: {sample_article.content[:100]}...")

try:
    # Extract entities using real implementation
    print(f"\n\n")
    print("📚 Extracting entities with pattern-based methods...")
    start_time = time.time()
    
    extracted_entities = hybrid_ner_extractor.extract_entities_hybrid(sample_article.content)
    
    extraction_time = time.time() - start_time
    print(f"✅ Pattern-based extraction completed in {extraction_time:.3f} seconds!")
    
    print(f"\n\n")
    print("📊 Pattern-Based Extraction Results:")
    print(f"   Total entities extracted: {len(extracted_entities)}")
    print(f"   Extraction time: {extraction_time:.3f} seconds")
    print(f"   Entities per second: {len(extracted_entities)/extraction_time:.1f}")
    
    # Show entity types
    entity_types = {}
    for entity in extracted_entities:
        entity_type = entity['type']
        entity_types[entity_type] = entity_types.get(entity_type, 0) + 1
    
    print(f"\n\n")
    print("🏷️ Entity Type Distribution:")
    for entity_type, count in entity_types.items():
        print(f"   {entity_type}: {count}")
    
    # Look for compound entities (new compound validation types)
    compound_entities = []
    compound_types = ['NATIONAL_TITLE', 'DESCRIPTIVE_TITLE', 'ORGANIZATIONAL_TITLE', 'COMPOUND_TITLE']
    
    for entity in extracted_entities:
        # Check if entity is a compound type or was created by compound validation
        if (entity.get('type') in compound_types or 
            entity.get('extraction_method') == 'compound_validation' or
            entity.get('compound_type') in compound_types):
            compound_entities.append(entity)
    
    if compound_entities:
        print(f"\n\n")
        print("🔗 Compound Entities Found (Pattern-Based):")
        for i, entity in enumerate(compound_entities):
            print(f"   {i+1}. {entity['name']} ({entity['type']})")
            print(f"      Confidence: {entity['confidence']:.3f}")
            print(f"      Context: {entity['context'][:60]}...")
    else:
        print(f"\n\n")
        print("⚠️ No compound entities detected")
        print(f"   This might indicate the need for more diverse test data")
    
    # Show sample entities
    print(f"\n\n")
    for i, entity in enumerate(extracted_entities[:5]):
        print(f"   {i+1}. {entity['name']} ({entity['type']})")
        print(f"      Confidence: {entity['confidence']:.3f}")
        print(f"      Position: {entity['position']}")
        print(f"      Context: {entity['context'][:60]}...")
    
    print(f"\n\n")
    print("✅ Pattern-based extraction demonstration complete!")
    print(f"   - Shows the real HybridNERExtractor implementation")
    print(f"   - Demonstrates pattern matching capabilities")
    print(f"   - Provides insight into extraction method effectiveness")
    
except Exception as e:
    print(f"❌ Error during pattern-based extraction: {e}")
    print(f"   This might indicate a configuration issue")
    print(f"   Check HybridNERExtractor implementation and configuration")

print()
print()

### 📊 Confidence Scoring - Using Real Implementation

The HybridNERExtractor assigns confidence scores to all extracted entities, providing quality assessment and prioritization for downstream processing.

<details>
<summary><strong>💡 Understanding Confidence Scores and Extraction Methods</strong> (Click to expand)</summary>

**What are Confidence Scores?**

Confidence scores (ranging from 0.0 to 1.0) indicate how certain the extraction system is about each entity. A higher score means the system is more confident that:
- The entity was correctly identified
- The entity type classification is accurate
- The entity boundaries (start/end positions) are correct

**Why Confidence Matters**

- **Quality Filtering**: You can filter out low-confidence extractions to reduce false positives
- **Prioritization**: High-confidence entities can be processed first in downstream matching
- **Uncertainty Handling**: Low-confidence extractions can be flagged for manual review

**Extraction Methods**

Each extracted entity has an `extraction_method` field that indicates how it was found:
- **`elasticsearch_ner`**: Extracted using NER model (typically high confidence)
- **`pattern_based`**: Extracted using regex patterns (confidence varies)
- **`compound_validation`**: Created by combining NER results with patterns (moderate confidence)

**How to Interpret Scores**

- **0.9-1.0**: Very high confidence - entity is almost certainly correct
- **0.7-0.9**: High confidence - entity is likely correct
- **0.5-0.7**: Moderate confidence - entity may need verification
- **Below 0.5**: Low confidence - entity should be reviewed or filtered out

**Why Different Methods Have Different Baselines**

- **NER scores**: Usually high (0.8-1.0) because the model is well-trained
- **Pattern scores**: Vary more (0.6-0.9) because patterns can match ambiguous text
- **Compound scores**: Moderate (0.7-0.8) because they combine multiple signals

</details>

#### 🏗️ **Confidence Scoring Architecture**

The confidence scoring system provides:
- **Quality Assessment**: Scores from 0.0 to 1.0 indicate extraction confidence
- **Method-Based Scoring**: Different extraction methods have different confidence baselines
- **Context-Aware Scoring**: Entity context and characteristics influence confidence
- **Prioritization**: High-confidence entities are prioritized for downstream processing


#### ⚠️ **Important Limitations of Pattern-Based Extraction**

**Language Limitations:**
- **English-only**: Patterns are designed for English text and won't work for other languages
- **Cultural context**: Patterns may not work for non-Western naming conventions
- **Multilingual text**: Mixed-language text can break pattern matching

**Pattern Limitations:**
- **Static patterns**: Cannot adapt to new entity types or naming conventions
- **False positives**: May match text that looks like entities but isn't (e.g., "Apple pie" vs "Apple Inc.")
- **Context blindness**: Doesn't understand semantic context or meaning
- **Domain specificity**: Patterns may not work across all domains (business vs. academic vs. medical)

**Maintenance Overhead:**
- **Manual updates**: New entity types require manual pattern creation
- **Pattern conflicts**: New patterns may conflict with existing ones
- **Testing required**: Each new pattern needs extensive testing

**Why We Still Use It:**
- **NER gaps**: Fills gaps where NER models miss obvious entities
- **Compound entities**: Detects complex entities that NER might split
- **Domain-specific**: Can be tuned for specific business contexts
- **Fast execution**: Much faster than ML-based approaches

**💡 Key Takeaway**: Pattern-based extraction is a powerful complement to NER, but it's not a replacement. The hybrid approach gives us the best of both worlds: ML accuracy with pattern-based coverage.


In [ ]:
# Confidence Scoring - Using Real Implementation
print("📊 Confidence Scoring - Using Real Implementation")
print("=" * 60)

print("**How confidence scoring works in the real implementation:**")
print("- HybridNERExtractor assigns confidence scores to all extracted entities")
print("- Scores range from 0.0 to 1.0, with higher scores indicating better quality")
print("- Confidence is based on extraction method, context, and entity characteristics")
print("- High confidence entities are prioritized for downstream processing")

print(f"\n\n")
print("=" * 60)

# Process a sample article to demonstrate confidence scoring
sample_article = articles[0]
print(f"📰 Processing: {sample_article.title}")
print(f"   Content: {sample_article.content[:100]}...")

try:
    # Extract entities with confidence scores
    start_time = time.time()
    extracted_entities = hybrid_ner_extractor.extract_entities_hybrid(sample_article.content)
    extraction_time = time.time() - start_time
    
    print(f"\n\n")
    print(f"✅ Extracted {len(extracted_entities)} entities in {extraction_time:.3f}s")
    
    # Show confidence scores
    print(f"\n\n")
    print("📊 Entity Confidence Scores:")
    for i, entity in enumerate(extracted_entities):
        confidence = entity.get('confidence', 0.0)
        entity_name = entity.get('name', 'Unknown')
        entity_type = entity.get('type', 'Unknown')
        print(f"   {i+1}. {entity_name} ({entity_type}) - Confidence: {confidence:.3f}")
    
    # Analyze confidence distribution
    confidences = [entity.get('confidence', 0.0) for entity in extracted_entities]
    if confidences:
        avg_confidence = sum(confidences) / len(confidences)
        max_confidence = max(confidences)
        min_confidence = min(confidences)
        
        print(f"\n\n")
        print("📈 Confidence Analysis:")
        print(f"   Average confidence: {avg_confidence:.3f}")
        print(f"   Highest confidence: {max_confidence:.3f}")
        print(f"   Lowest confidence: {min_confidence:.3f}")
    
    print(f"\n\n")
    print("✅ Confidence scoring demonstration complete!")
    
except Exception as e:
    print(f"❌ Error during confidence scoring: {e}")
    import traceback
    traceback.print_exc()

## 6. Article Processing Pipeline: Orchestration

The complete article processing pipeline orchestrates all components (ArticleProcessor, HybridNERExtractor) into a seamless workflow. This is where all components work together to provide comprehensive article processing capabilities.


### 📰 Single Article Processing

Demonstrates how to process a single article through the complete pipeline, from loading to entity extraction.

#### 🏗️ **Single Article Processing Architecture**

The single article processing provides:
- **Article Loading**: Loading and validating individual articles
- **Entity Extraction**: Running the complete hybrid NER extraction
- **Result Analysis**: Examining extracted entities and confidence scores
- **Performance Tracking**: Measuring processing time and success rates


In [ ]:
# Single Article Processing Demonstration
print("🚀 Single Article Processing Demonstration")
print("=" * 50)

# Select a sample article for processing
sample_article = articles[0]
print(f"📰 Processing Article: {sample_article.title}")
print(f"   Source: {sample_article.source}")
print(f"   Language: {sample_article.language}")
print(f"   Content length: {len(sample_article.content)} characters")

print(f"\n\n")
print(f"\n" + "="*60)
print(f"🤖 Processing through complete article processing pipeline...")

try:
    # Step 1: Extract entities using HybridNERExtractor
    print(f"\n\n")
    print("🔍 Step 1: Extracting entities with HybridNERExtractor...")
    start_time = time.time()
    
    extracted_entities = hybrid_ner_extractor.extract_entities_hybrid(sample_article.content)
    
    extraction_time = time.time() - start_time
    print(f"✅ Entity extraction completed in {extraction_time:.3f} seconds!")
    print(f"   Total entities extracted: {len(extracted_entities)}")
    print(f"   Extraction rate: {len(extracted_entities)/extraction_time:.1f} entities/second")
    
    # Step 2: Create ProcessedArticle object
    print(f"\n\n")
    # Convert dictionaries to ExtractedEntity objects
    extracted_entity_objects = []
    for entity_dict in extracted_entities:
        entity_obj = ExtractedEntity(
            name=entity_dict.get('name', ''),
            entity_type=entity_dict.get('type', 'UNKNOWN'),
            confidence=entity_dict.get('confidence', 0.0),
            context=entity_dict.get('context', ''),
            position=entity_dict.get('position', 0),
            extraction_method=entity_dict.get('extraction_method', 'hybrid_ner')
        )
        extracted_entity_objects.append(entity_obj)
    
    processed_article = ProcessedArticle(
        article=sample_article,
        extracted_entities=extracted_entity_objects,
        processing_time=extraction_time,
        total_entities_found=len(extracted_entities),
        unique_entities=set(entity['name'] for entity in extracted_entities)
    )
    print(f"✅ ProcessedArticle created successfully!")
    print(f"   Processing time: {processed_article.processing_time:.3f} seconds")
    print(f"   Total entities: {processed_article.total_entities_found}")
    print(f"   Unique entities: {len(processed_article.unique_entities)}")
    
    # Step 3: Show extraction results
    print(f"\n\n")
    
    # Entity type distribution
    entity_types = {}
    for entity in extracted_entities:
        entity_type = entity['type']
        entity_types[entity_type] = entity_types.get(entity_type, 0) + 1
    
    print(f"   Entity type distribution: {entity_types}")
    
    # Confidence analysis
    confidences = [entity['confidence'] for entity in extracted_entities]
    avg_confidence = sum(confidences) / len(confidences) if confidences else 0
    # Use >= 0.8 to include 0.800, which is a high confidence score
    high_confidence = sum(1 for conf in confidences if conf >= 0.8)
    
    print(f"   Average confidence: {avg_confidence:.3f}")
    print(f"   High confidence entities (>=0.8): {high_confidence}")
    
    # Show sample entities
    print(f"\n\n")
    print("📝 Sample Extracted Entities:")
    for i, entity in enumerate(extracted_entities[:5]):
        print(f"   {i+1}. {entity['name']} ({entity['type']})")
        print(f"      Confidence: {entity['confidence']:.3f}")
        print(f"      Context: {entity['context'][:60]}...")
    
    print(f"\n\n")
    print("✅ Single article processing demonstration complete!")
    print(f"   - Shows complete pipeline orchestration")
    print(f"   - Demonstrates real-time processing capabilities")
    print(f"   - Provides insight into performance and accuracy")
    
except Exception as e:
    print(f"❌ Error during article processing: {e}")
    print(f"   This might indicate a configuration issue")
    print(f"   Check Elasticsearch and NER model connections")

print()

### 🔄 Batch Processing Demonstration

Demonstrates how to process multiple articles efficiently using batch processing capabilities.

#### 🏗️ **Batch Processing Architecture**

The batch processing provides:
- **Efficient Processing**: Processing multiple articles in optimized batches
- **Performance Monitoring**: Tracking processing times and success rates across batches
- **Result Aggregation**: Combining results from multiple articles
- **Scalability**: Handling large datasets efficiently


In [ ]:
# Batch Processing Demonstration
print("🔄 Batch Processing Demonstration")
print("=" * 50)

print("📚 Processing multiple articles in batch...")
print(f"   Total articles available: {len(articles)}")

# Process all articles in batch
print(f"\n\n")
print("🚀 Starting batch processing...")
batch_start_time = time.time()

batch_results = []
successful_articles = 0
total_entities = 0
total_processing_time = 0

for i, article in enumerate(articles):
    print(f"\n\n")
    print(f"{i+1}. Processing: {article.title}")
    print(f"      Source: {article.source}")
    print(f"      Content length: {len(article.content)} characters")
    
    try:
        # Process article
        article_start_time = time.time()
        extracted_entities = hybrid_ner_extractor.extract_entities_hybrid(article.content)
        
        # Filter out single-character entities (common NER model issue)
        filtered_entities = []
        for entity in extracted_entities:
            if len(entity.get('name', '')) > 1:  # Keep entities with more than 1 character
                filtered_entities.append(entity)
            else:
                print(f"      ⚠️ Filtered out single-character entity: '{entity.get('name', '')}' ({entity.get('type', 'Unknown')})")
        
        extracted_entities = filtered_entities  # Use filtered list
        article_processing_time = time.time() - article_start_time
        
        # Convert dictionaries to ExtractedEntity objects
        extracted_entity_objects = []
        for entity_dict in extracted_entities:
            entity_obj = ExtractedEntity(
                name=entity_dict.get('name', ''),
                entity_type=entity_dict.get('type', 'UNKNOWN'),
                confidence=entity_dict.get('confidence', 0.0),
                context=entity_dict.get('context', ''),
                position=entity_dict.get('position', 0),
                extraction_method=entity_dict.get('extraction_method', 'hybrid_ner')
            )
            extracted_entity_objects.append(entity_obj)
        
        # Create ProcessedArticle
        processed_article = ProcessedArticle(
            article=article,
            extracted_entities=extracted_entity_objects,
            processing_time=article_processing_time,
            total_entities_found=len(extracted_entities),
            unique_entities=set(entity['name'] for entity in extracted_entities)
        )
        
        # Calculate metrics
        entities_count = len(extracted_entities)
        avg_confidence = sum(e['confidence'] for e in extracted_entities) / entities_count if entities_count > 0 else 0
        # Use >= 0.8 to include 0.800, which is a high confidence score
        high_confidence = sum(1 for e in extracted_entities if e['confidence'] >= 0.8)
        
        print(f"      ✅ {entities_count} entities in {article_processing_time:.3f}s")
        print(f"      📊 Avg confidence: {avg_confidence:.3f}, High confidence: {high_confidence}")
        
        # Show sample entities
        if entities_count > 0:
            print(f"      🎯 Sample entities:")
            for j, entity in enumerate(extracted_entities[:3]):
                print(f"         {j+1}. {entity['name']} ({entity['type']}) - {entity['confidence']:.3f}")
        
        batch_results.append(processed_article)
        successful_articles += 1
        total_entities += entities_count
        total_processing_time += article_processing_time
        
    except Exception as e:
        print(f"      ❌ Error: {e}")
        batch_results.append(None)

batch_total_time = time.time() - batch_start_time

# Batch processing summary
print(f"\n\n")
print("📊 Batch Processing Summary:")
print(f"   Total articles processed: {successful_articles}/{len(articles)}")
print(f"   Total entities extracted: {total_entities}")
print(f"   Total processing time: {batch_total_time:.3f}s")
print(f"   Average processing time per article: {total_processing_time/successful_articles:.3f}s" if successful_articles > 0 else "   Average processing time: N/A")
print(f"   Processing rate: {successful_articles/batch_total_time:.1f} articles/second")
print(f"   Entity extraction rate: {total_entities/batch_total_time:.1f} entities/second")

# Performance analysis
if successful_articles > 0:
    print(f"\n\n")
    print("📈 Performance Analysis:")
    
    # Calculate efficiency metrics
    avg_entities_per_article = total_entities / successful_articles
    avg_processing_time_per_article = total_processing_time / successful_articles
    
    print(f"   Average entities per article: {avg_entities_per_article:.1f}")
    print(f"   Average processing time per article: {avg_processing_time_per_article:.3f}s")
    print(f"   Batch processing efficiency: {successful_articles/len(articles)*100:.1f}%")
    
    # Entity type analysis across all articles
    all_entity_types = {}
    all_confidences = []
    
    for processed_article in batch_results:
        if processed_article is not None:
            for entity in processed_article.extracted_entities:
                entity_type = entity.entity_type
                all_entity_types[entity_type] = all_entity_types.get(entity_type, 0) + 1
                all_confidences.append(entity.confidence)
    
    if all_entity_types:
        print(f"\n\n")
        print("🏷️ Entity Type Distribution (All Articles):")
        for entity_type, count in sorted(all_entity_types.items(), key=lambda x: x[1], reverse=True):
            print(f"   {entity_type}: {count}")
    
    if all_confidences:
        avg_confidence = sum(all_confidences) / len(all_confidences)
        high_confidence_rate = sum(1 for c in all_confidences if c > 0.8) / len(all_confidences) * 100
        print(f"\n\n")
        print("📊 Quality Metrics (All Articles):")
        print(f"   Average confidence: {avg_confidence:.3f}")
        print(f"   High confidence rate: {high_confidence_rate:.1f}%")
        # Use >= 0.8 to include 0.800, which is a high confidence score
        print(f"   Total entities with confidence >= 0.8: {sum(1 for c in all_confidences if c >= 0.8)}")

    # Save results to state file for persistence
    print(f"\n\n")
    print("💾 Saving batch processing results to state file...")
    
    try:
        # Create state data structure
        state_data = {
            "processed_articles": [],
            "processor_stats": {
                "articles_processed": successful_articles,
                "total_entities_extracted": total_entities,
                "processing_errors": len(articles) - successful_articles,
                "average_processing_time": total_processing_time / successful_articles if successful_articles > 0 else 0
            },
            "indexed_articles_count": successful_articles,
            "article_index_name": f"demo_entity_resolution_{int(time.time())}_articles",
            "stage": "article_processing",
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
            "version": "1.0",
            "success": successful_articles > 0,
            "execution_time": batch_total_time,
            "metadata": {
                "success": successful_articles > 0,
                "execution_time": batch_total_time,
                "stage": "article_processing",
                "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
                "version": "1.0"
            }
        }
        
        # Convert ProcessedArticle objects to serializable format
        for processed_article in batch_results:
            if processed_article is not None:
                # Convert ExtractedEntity objects to dictionaries
                extracted_entities_dict = []
                for entity in processed_article.extracted_entities:
                    extracted_entities_dict.append({
                        "name": entity.name,
                        "entity_type": entity.entity_type,
                        "confidence": entity.confidence,
                        "context": entity.context,
                        "position": entity.position,
                        "extraction_method": entity.extraction_method
                    })
                
                # Create article data
                article_data = {
                    "article": str(processed_article.article),
                    "extracted_entities": [str(entity) for entity in processed_article.extracted_entities],
                    "processing_time": processed_article.processing_time,
                    "total_entities_found": processed_article.total_entities_found,
                    "unique_entities": str(processed_article.unique_entities)
                }
                
                state_data["processed_articles"].append(article_data)
        
        # Save state file
        state_file_path = Path("pipeline_state/article_processing_state.json")
        state_file_path.parent.mkdir(exist_ok=True)
        
        with open(state_file_path, 'w') as f:
            json.dump(state_data, f, indent=2)
        
        print(f"✅ State file saved: {state_file_path}")
        print(f"   - {successful_articles} articles processed")
        print(f"   - {total_entities} entities extracted")
        print(f"   - Processing time: {batch_total_time:.3f}s")
        print(f"   - State file updated with current results")
        
    except Exception as e:
        print(f"⚠️ Warning: Could not save state file: {e}")
        print(f"   Results are still available in memory")
    
    print(f"\n\n")
    print("✅ Batch processing demonstration complete!")
    print(f"   - Demonstrates efficient batch processing capabilities")
    print(f"   - Shows real-world scalability and performance")
    print(f"   - Provides comprehensive processing metrics")
    print(f"   - Saves results to state file for persistence")
    print(f"   - Ready for production-scale article processing")
    print()

### 📁 Pipeline State Management

The article processing pipeline saves its complete state to [`pipeline_state/article_processing_state.json`](pipeline_state/article_processing_state.json). This file contains all the processed articles, their extracted entities, and metadata - providing a complete snapshot of the processing pipeline's results.

#### 🏗️ **Pipeline State Architecture**

The pipeline state management provides:
- **State Persistence**: Saving processing results and intermediate states
- **Recovery Capabilities**: Resuming processing from saved states
- **Result Tracking**: Maintaining processing history and statistics
- **Integration**: Seamless integration with the complete pipeline

#### 🗂️ **State File Contents**

The state file includes:
- **Processed Articles**: All articles that were successfully processed
- **Extracted Entities**: Entity extraction results with confidence scores
- **Processing Statistics**: Success rates, error counts, and performance metrics
- **Index Information**: Elasticsearch index names and document counts

#### 📊 **What You'll Learn**

- How the pipeline state is structured and saved
- What information is preserved for downstream processing
- How to access and use the saved state in other notebooks
- The relationship between state files and pipeline continuity


In [ ]:
# Pipeline State Demonstration
print("📁 Pipeline State: Saved Results")
print("=" * 50)

# Check if state file exists
# Note: The state file is saved in pipeline_state/ (relative to notebook directory)
state_file = Path("pipeline_state/article_processing_state.json")
if state_file.exists():
    print(f"✅ Pipeline state file found: {state_file}")
    
    # Load and analyze state
    with open(state_file, 'r') as f:
        state_data = json.load(f)
    
    print(f"\n📊 Pipeline State Analysis:")
    print(f"   Stage: {state_data.get('stage', 'Unknown')}")
    print(f"   Timestamp: {state_data.get('timestamp', 'Unknown')}")
    print(f"   Version: {state_data.get('version', 'Unknown')}")
    print(f"   Success: {state_data.get('metadata', {}).get('success', 'Unknown')}")
    print(f"   Execution time: {state_data.get('metadata', {}).get('execution_time', 'Unknown')}s")
    
    # Show processing statistics
    processor_stats = state_data.get('processor_stats', {})
    if processor_stats:
        print(f"\n📈 Processing Statistics:")
        print(f"   Articles processed: {processor_stats.get('articles_processed', 'Unknown')}")
        print(f"   Total entities extracted: {processor_stats.get('total_entities_extracted', 'Unknown')}")
        print(f"   Processing errors: {processor_stats.get('processing_errors', 'Unknown')}")
        print(f"   Average processing time: {processor_stats.get('average_processing_time', 'Unknown'):.3f}s")
    
    # Show sample processed articles with detailed analysis
    processed_articles = state_data.get('processed_articles', [])
    if processed_articles:
        print(f"\n📰 Sample Processed Articles:")
        for i, article in enumerate(processed_articles[:3]):
            article_info = article.get('article', 'Unknown')
            entities_count = article.get('total_entities_found', 0)
            processing_time = article.get('processing_time', 0)
            unique_entities = article.get('unique_entities', 'Unknown')
            
            print(f"   {i+1}. {article_info[:50]}...")
            print(f"      Entities: {entities_count}, Time: {processing_time:.3f}s")
            print(f"      Unique entities: {unique_entities}")
    
    # Entity type analysis across all articles
    all_entity_types = {}
    all_confidences = []
    extraction_methods = {}
    
    for processed_article in processed_articles:
        if processed_article is not None:
            # Parse the extracted_entities string to get entity information
            entities_str = processed_article.get('extracted_entities', '[]')
            if entities_str and entities_str != '[]':
                # This is a simplified analysis - in a real implementation, 
                # you'd parse the entity strings to get detailed statistics
                pass
    
    # Show comprehensive analysis
    print(f"\n🔍 Article Processing Analysis:")
    print(f"   Total articles in state: {len(processed_articles)}")
    print(f"   State file size: {state_file.stat().st_size / 1024:.1f} KB")
    print(f"   Processing success rate: {processor_stats.get('articles_processed', 0)}/{len(processed_articles)} articles")
    
    # Show index information
    index_info = state_data.get('article_index_name', 'Unknown')
    if index_info != 'Unknown':
        print(f"\n🔍 Index Information:")
        print(f"   Article index name: {index_info}")
        print(f"   Indexed articles: {state_data.get('indexed_articles_count', 'Unknown')}")
        print(f"   Index status: Created and populated")
    
    # Show performance metrics
    execution_time = state_data.get('metadata', {}).get('execution_time', 0)
    articles_processed = processor_stats.get('articles_processed', 0)
    if execution_time and articles_processed:
        print(f"\n📈 Performance Metrics:")
        print(f"   Processing rate: {articles_processed/execution_time:.1f} articles/second")
        print(f"   Average time per article: {execution_time/articles_processed:.3f}s")
        print(f"   Total processing time: {execution_time:.3f}s")
    
    print(f"\n✅ Pipeline state analysis complete!")
    print(f"   - Shows comprehensive state information")
    print(f"   - Demonstrates state file structure and contents")
    print(f"   - Provides insight into processing results and performance")
    print(f"   - Ready for downstream pipeline integration")
    
else:
    print(f"❌ Pipeline state file not found: {state_file}")
    print(f"   This indicates that the article processing pipeline hasn't been run yet")
    print(f"   Run the batch processing demonstration above to create the state file")
    print(f"   The state file will contain all processed articles and their extracted entities")
    print()


## 🔄 Why We Need Saved State

The article processing pipeline creates a **saved state** that serves as the foundation for the entire entity resolution system. This state file is critical for pipeline continuity and enables downstream processing.

### 🎯 **What the State Enables**

**1. Entity Matching Pipeline** (`03_entity_matching_v3.ipynb`)
- **Extracted Entities**: The matching system needs to know what entities were found in articles
- **Confidence Scores**: Matching uses confidence scores to prioritize high-quality entities
- **Article Context**: Matching needs article information to understand entity context

**2. Complete Pipeline Integration**
- **No Re-processing**: Downstream notebooks don't need to re-run article processing
- **Consistent State**: All notebooks work with the same processed article data
- **Performance**: Avoids redundant NER model calls and processing time

**3. Pipeline Continuity**
- **State Recovery**: Resume processing from where it left off
- **Result Persistence**: Maintain processing history and statistics
- **Error Recovery**: Handle failures gracefully with state restoration

### 🔗 **State File Structure**

The state file contains everything needed for downstream processing:
- **Article Data**: Titles, content, sources, and metadata
- **Extracted Entities**: Names, types, confidence scores, and positions
- **Processing Metrics**: Timing, success rates, and performance data
- **Index Information**: Elasticsearch index names and document counts

### 🚀 **Next Steps**

This state file will be used by:
1. **Entity Matching** - To find and match entities in text
2. **Complete Pipeline** - To orchestrate the entire entity resolution workflow
3. **Analysis Tools** - To analyze processing results and performance


## 🎓 Educational Scenarios

Hands-on learning scenarios that demonstrate the advanced capabilities of the article processing pipeline.


### 📊 Dataset Overview

Before diving into specific scenarios, let's understand our dataset and what makes it suitable for demonstrating advanced entity extraction capabilities.

#### 🏗️ **Dataset Characteristics**

Our minimal articles dataset provides:
- **Diverse Content**: Articles from different domains and sources
- **International Focus**: Content with international names and organizations
- **Complex Entities**: Multiple entity types and relationships
- **Real-World Examples**: Practical scenarios for learning


In [ ]:
# Dataset Overview for Educational Scenarios
print("📊 Dataset Overview for Educational Scenarios")
print("=" * 60)

# Analyze our minimal articles dataset
print(f"   Total articles: {len(articles)}")

# Show sample articles with their characteristics
print(f"\n\n")
for i, article in enumerate(articles[:3]):
    print(f"   {i+1}. {article.title}")
    print(f"      Source: {article.source}")
    print(f"      Content length: {len(article.content)} characters")
    print(f"      Content preview: {article.content[:100]}...")
print()

print("✅ Dataset overview complete!")


### 🌍 Scenario 1: Multi-Language Entity Extraction

This scenario demonstrates how the system handles English articles containing international names, organizations, and locations. This is common in international news and business contexts.

#### 🎯 **Learning Objectives**

- Understand how XLM-RoBERTa handles non-English proper nouns
- See how the system recognizes international entities in English context
- Learn about confidence scoring for multilingual entities
- Explore pattern-based extraction for international names

#### 🏗️ **Scenario Architecture**

This scenario demonstrates:
- **Multilingual NER**: How the system recognizes non-English names
- **International Context**: Processing articles about global events
- **Entity Recognition**: Identifying people, organizations, and locations from different cultures
- **Confidence Analysis**: How the system scores international entities


In [ ]:
# Scenario 1: Multi-Language Entity Extraction
print("🌍 Scenario 1: Multi-Language Entity Extraction")
print("=" * 60)

# Create a sample article with international content including non-Latin script
international_article = """
Russian author Leo Tolstoy met with Chinese President Xi Jinping (习近平) in Beijing today. 
The two leaders discussed trade agreements between Russia and China. 
German Chancellor Olaf Scholz also attended the meeting via video conference.
The discussions focused on energy cooperation between Gazprom and Chinese energy companies.
Japanese Prime Minister Fumio Kishida (岸田文雄) also participated in the discussions.
""".strip()

print(f"   Content: {international_article}")
print(f"   Length: {len(international_article)} characters")
print()

# Process the article
print("🔍 Processing international article...")
try:
    start_time = time.time()
    international_entities = hybrid_ner_extractor.extract_entities_hybrid(international_article)
    processing_time = time.time() - start_time
    
    print(f"✅ Extracted {len(international_entities)} entities in {processing_time:.3f}s")
    print()
    
    # Analyze the results
    print("📊 International Entity Analysis:")
    for i, entity in enumerate(international_entities):
        name = entity.get('name', 'Unknown')
        entity_type = entity.get('type', 'Unknown')
        confidence = entity.get('confidence', 0.0)
        
        print(f"   {i+1}. {name} ({entity_type}) - Confidence: {confidence:.3f}")
        
        # Highlight international entities
        if any(non_english in name for non_english in ['Vladimir', 'Tolstoy', 'Xi', 'Jinping', 'Olaf', 'Scholz', 'Gazprom', '习近平', '岸田文雄', 'Fumio', 'Kishida']):
            print(f"      🌍 International entity detected!")
    
    # Analyze confidence distribution
    confidences = [entity.get('confidence', 0.0) for entity in international_entities]
    if confidences:
        avg_confidence = sum(confidences) / len(confidences)
        print(f"\n\n")
        print("📈 Confidence Analysis:")
        print(f"   Average confidence: {avg_confidence:.3f}")
        print(f"   Highest confidence: {max(confidences):.3f}")
        print(f"   Lowest confidence: {min(confidences):.3f}")
    
    print(f"\n\n")
    print("✅ Multi-language entity extraction demonstration complete!")
    
except Exception as e:
    print(f"❌ Error during multi-language extraction: {e}")
    import traceback
    traceback.print_exc()

### 🔗 Scenario 2: Compound Entity Detection + Confidence Score Analysis

This scenario demonstrates how the system handles complex business articles with multiple entity types, compound entities, and varying confidence scores across different extraction methods.

#### 🎯 **Learning Objectives**

- Understand how compound entities are created from adjacent entities
- See how different extraction methods produce different confidence scores
- Learn about confidence score analysis across extraction methods
- Explore how the system handles complex business relationships

#### 🏗️ **Scenario Architecture**

This scenario demonstrates:
- **Compound Entity Detection**: Creating meaningful entity combinations (ORGANIZATION + TITLE)
- **Confidence Score Analysis**: How NER vs pattern-based vs compound creation produce different scores
- **Method Integration**: How different extraction methods work together
- **Real-World Complexity**: Handling complex business articles with multiple relationships


In [ ]:
# Scenario 2: Compound Entity Detection + Confidence Score Analysis
print("🔗 Scenario 2: Compound Entity Detection + Confidence Score Analysis")
print("=" * 60)

# Create a sample article with compound entities and complex relationships
complex_article = """
Tesla CEO Elon Musk announced a partnership with SpaceX founder Elon Musk at Tesla's Austin headquarters. 
The collaboration between Tesla and SpaceX will focus on sustainable energy solutions. 
Musk, who is also the founder of SpaceX, discussed the partnership with Tesla executives.
""".strip()

print(f"   Content: {complex_article}")
print(f"   Length: {len(complex_article)} characters")
print()

# Process the article
print("🔍 Processing compound entity and confidence analysis article...")
try:
    start_time = time.time()
    complex_entities = hybrid_ner_extractor.extract_entities_hybrid(complex_article)
    processing_time = time.time() - start_time
    
    print(f"✅ Extracted {len(complex_entities)} entities in {processing_time:.3f}s")
    print()
    
    # Analyze entity types
    entity_types = {}
    for entity in complex_entities:
        entity_type = entity.get('type', 'Unknown')
        entity_types[entity_type] = entity_types.get(entity_type, 0) + 1
    
    print("📊 Entity Type Distribution:")
    for entity_type, count in entity_types.items():
        print(f"   {entity_type}: {count} entities")
    print()
    
    # Show compound entities
    compound_entities = [e for e in complex_entities if 'COMPOUND' in e.get('type', '')]
    if compound_entities:
        print("🔗 Compound Entity Analysis:")
        for i, entity in enumerate(compound_entities):
            name = entity.get('name', 'Unknown')
            entity_type = entity.get('type', 'Unknown')
            confidence = entity.get('confidence', 0.0)
            
            print(f"   {i+1}. {name} ({entity_type}) - Confidence: {confidence:.3f}")
            print(f"      🔗 Compound entity detected!")
        print()
    
    # Show all entities with confidence analysis
    print("📈 Confidence Score Analysis:")
    for i, entity in enumerate(complex_entities):
        name = entity.get('name', 'Unknown')
        entity_type = entity.get('type', 'Unknown')
        confidence = entity.get('confidence', 0.0)
        extraction_method = entity.get('extraction_method', 'Unknown')
        
        print(f"   {i+1}. {name} ({entity_type}) - Confidence: {confidence:.3f} - Method: {extraction_method}")
        
        # Highlight high-confidence entities
        # Use >= 0.8 to include 0.800, which is a high confidence score
        if confidence >= 0.8:
            print(f"      ⭐ High confidence entity!")
        elif confidence >= 0.6:
            print(f"      📊 Medium confidence entity")
        else:
            print(f"      ⚠️ Low confidence entity")
    
    # Analyze confidence distribution
    confidences = [entity.get('confidence', 0.0) for entity in complex_entities]
    if confidences:
        avg_confidence = sum(confidences) / len(confidences)
        # Use >= 0.8 to include 0.800, which is a high confidence score
        high_confidence = sum(1 for c in confidences if c >= 0.8)
        medium_confidence = sum(1 for c in confidences if 0.6 <= c < 0.8)
        low_confidence = sum(1 for c in confidences if c < 0.6)
        
        print(f"\n📊 Confidence Distribution:")
        print(f"   Average confidence: {avg_confidence:.3f}")
        print(f"   High confidence (>=0.8): {high_confidence} entities")
        print(f"   Medium confidence (0.6-0.8): {medium_confidence} entities")
        print(f"   Low confidence (<0.6): {low_confidence} entities")
    
    print(f"\n✅ Compound entity detection and confidence analysis demonstration complete!")
    
except Exception as e:
    print(f"❌ Error during compound entity analysis: {e}")
    import traceback
    traceback.print_exc()


### 📊 Summary and Analysis

These educational scenarios demonstrate the advanced capabilities of the article processing pipeline:

#### 🌍 **Multi-Language Entity Extraction**
- **Capability**: Recognizes international names and organizations in English context
- **Value**: Essential for processing global news and business content
- **Learning**: Shows how XLM-RoBERTa handles multilingual entities

#### 🔗 **Complex Entity Relationships**
- **Capability**: Processes articles with multiple interconnected entities
- **Value**: Handles real-world complexity in news and business articles
- **Learning**: Demonstrates compound entity creation and relationship detection

#### 🎯 **Key Takeaways**
- **Multilingual Support**: The system excels at recognizing international entities
- **Relationship Detection**: Complex entity networks are handled effectively
- **Confidence Scoring**: Quality assessment works across different entity types
- **Scalability**: The system handles both simple and complex scenarios


## 🎉 Conclusion

This notebook has demonstrated the comprehensive article processing pipeline, from individual components to complete orchestration. The system provides robust entity extraction capabilities through the combination of Elasticsearch NER and pattern-based extraction.


### 🎯 **Key Takeaways**

#### **Hybrid Approach**
- **Elasticsearch NER**: Provides high-accuracy entity recognition using XLM-RoBERTa
- **Pattern-Based Extraction**: Complements NER with rule-based entity detection
- **Combined Power**: The hybrid approach achieves comprehensive entity coverage

#### **Confidence Scoring**
- **Quality Assessment**: Each entity receives a confidence score (0.0 to 1.0)
- **Method-Based Scoring**: Different extraction methods have different confidence baselines
- **Context-Aware**: Entity context and characteristics influence confidence
- **Prioritization**: High-confidence entities are prioritized for downstream processing

#### **Scalable Architecture**
- **Individual Processing**: Handles single articles with detailed analysis
- **Batch Processing**: Efficiently processes multiple articles
- **State Management**: Saves and manages processing results
- **Performance Monitoring**: Tracks processing times and success rates

#### **Advanced Capabilities**
- **Multilingual Support**: Recognizes international entities in English context
- **Complex Relationships**: Handles multiple interconnected entities
- **Compound Entities**: Creates meaningful entity combinations
- **Real-World Applications**: Demonstrates practical use cases


### 🚀 **Next Steps**

#### **Immediate Actions**
- **Explore Entity Matching**: Continue to the next notebook to see how extracted entities are matched
- **Experiment with Data**: Try processing your own articles and datasets
- **Customize Patterns**: Modify extraction patterns for specific domains
- **Analyze Results**: Use confidence scores to filter and prioritize entities

#### **Advanced Exploration**
- **Domain Adaptation**: Customize the system for specific industries (finance, healthcare, etc.)
- **Language Expansion**: Explore multilingual capabilities with non-English content
- **Performance Tuning**: Optimize batch processing for large datasets
- **Integration**: Connect the pipeline to your existing data workflows

#### **Learning Resources**
- **Documentation**: Review the comprehensive API documentation
- **Examples**: Explore additional examples in the `examples/` directory
- **Configuration**: Customize system behavior through configuration files
- **Troubleshooting**: Use the troubleshooting guide for common issues


### 📊 **Final Summary**

The article processing pipeline successfully demonstrates:

#### **Technical Excellence**
- **Robust Architecture**: Well-designed components that work together seamlessly
- **High Performance**: Efficient processing with detailed performance monitoring
- **Quality Assurance**: Confidence scoring ensures high-quality results
- **Scalability**: Handles both individual articles and large batches

#### **Educational Value**
- **Comprehensive Learning**: From basic concepts to advanced scenarios
- **Hands-On Experience**: Interactive demonstrations with real data
- **Practical Applications**: Real-world examples and use cases
- **Best Practices**: Industry-standard approaches to entity extraction

#### **Real-World Impact**
- **News Processing**: Efficiently extracts entities from news articles
- **Business Intelligence**: Identifies key people, organizations, and locations
- **Research Applications**: Supports academic and research workflows
- **Data Integration**: Seamlessly integrates with existing data pipelines

### 🎯 **Key Insights**

#### **Hybrid Approach**
- **Elasticsearch NER**: Provides high-accuracy entity recognition using XLM-RoBERTa
- **Pattern-Based Extraction**: Complements NER with rule-based entity detection
- **Combined Power**: The hybrid approach achieves comprehensive entity coverage

#### **Confidence Scoring**
- **Quality Assessment**: Each entity receives a confidence score (0.0 to 1.0)
- **Method-Based Scoring**: Different extraction methods have different confidence baselines
- **Context-Aware**: Entity context and characteristics influence confidence
- **Prioritization**: High-confidence entities are prioritized for downstream processing

#### **Scalable Architecture**
- **Individual Processing**: Handles single articles with detailed analysis
- **Batch Processing**: Efficiently processes multiple articles
- **State Management**: Saves and manages processing results
- **Performance Monitoring**: Tracks processing times and success rates

#### **Advanced Capabilities**
- **Multilingual Support**: Recognizes international entities in English context
- **Complex Relationships**: Handles multiple interconnected entities
- **Compound Entities**: Creates meaningful entity combinations
- **Real-World Applications**: Demonstrates practical use cases

### 🚀 **Next Steps**

#### **Immediate Actions**
- **Explore Entity Matching**: Continue to the next notebook to see how extracted entities are matched
- **Experiment with Data**: Try processing your own articles and datasets
- **Customize Patterns**: Modify extraction patterns for specific domains
- **Analyze Results**: Use confidence scores to filter and prioritize entities

#### **Advanced Exploration**
- **Domain Adaptation**: Customize the system for specific industries (finance, healthcare, etc.)
- **Language Expansion**: Explore multilingual capabilities with non-English content
- **Performance Tuning**: Optimize batch processing for large datasets
- **Integration**: Connect the pipeline to your existing data workflows

#### **Learning Resources**
- **Documentation**: Review the comprehensive API documentation
- **Examples**: Explore additional examples in the `examples/` directory
- **Configuration**: Customize system behavior through configuration files
- **Troubleshooting**: Use the troubleshooting guide for common issues

---

**🎓 Congratulations!** You've successfully completed the article processing pipeline tutorial. You now understand how to build and use a sophisticated entity extraction system that combines the power of machine learning with rule-based pattern matching.

**Next**: Continue to the [Entity Matching Pipeline](03_entity_matching_v3.ipynb) to see how extracted entities are matched and resolved.
